# 🧠 Intel AI Boost NPU 실행 — 환자안전 모니터링

**OpenVINO** + **NNCF INT8 양자화**를 사용해 Intel AI Boost NPU에서
Fine-tuned MobileVLM V2를 실행합니다.

| 항목 | 내용 |
|---|---|
| 백엔드 | OpenVINO Runtime (NPU Plugin) |
| 대상 하드웨어 | Intel AI Boost NPU (Core Ultra 시리즈) |
| 예상 속도 | 1~3초/장 |
| 양자화 | INT8 (NNCF) |

> ⚠️ **실행 순서**: 먼저 `inference_arc_gpu.ipynb` Cell 3에서 LoRA 병합을 완료해야 합니다.  
> ⚠️ **드라이버**: NPU 드라이버 필요 — https://www.intel.com/content/www/us/en/download/794734/intel-npu-driver-windows.html

## 📦 Cell 1: OpenVINO / NNCF 설치 (최초 1회)

In [ ]:
import subprocess

packages = [
    'openvino>=2024.0',        # OpenVINO Runtime
    'nncf',                    # 양자화 (INT8)
    'optimum[openvino]',       # Optimum Intel
    'onnx',                    # ONNX Export
    'onnxruntime',             # ONNX Runtime
    'transformers', 'peft', 'sentencepiece',
    'timm', 'torchvision', 'pillow',
    'opencv-python', 'einops', 'accelerate', 'safetensors'
]

for pkg in packages:
    result = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True)
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print('\n⚠️ 커널 재시작 후 Cell 2부터 실행하세요!')

## ⚙️ Cell 2: NPU 확인 & 경로 설정

In [ ]:
import os, sys, time, cv2
from datetime import datetime
from pathlib import Path
from PIL import Image

# ── NPU 확인 ─────────────────────────────────────────────────
try:
    import openvino as ov
    core = ov.Core()
    available_devices = core.available_devices
    print(f'✅ OpenVINO {ov.__version__}')
    print(f'   사용 가능 디바이스: {available_devices}')

    if 'NPU' in available_devices:
        TARGET_DEVICE = 'NPU'
        print('   ✅ NPU 감지됨 → NPU 실행')
    elif 'GPU' in available_devices:
        TARGET_DEVICE = 'GPU'
        print('   ⚠️ NPU 없음 → GPU(ARC) 폴백')
    else:
        TARGET_DEVICE = 'CPU'
        print('   ⚠️ NPU/GPU 없음 → CPU 폴백')
except ImportError:
    print('❌ OpenVINO 미설치 → Cell 1 먼저 실행')
    TARGET_DEVICE = 'CPU'

# ── 경로 설정 ─────────────────────────────────────────────────
BASE_DIR      = Path(r'c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM')
MERGED_DIR    = Path(r'C:\mobilevlm_merged')          # ⚠️ 한글 없는 경로로 수정됨
OV_DIR        = Path(r'C:\mobilevlm_openvino')        # ⚠️ 한글 없는 경로로 수정됨
VIDEO_DIR     = BASE_DIR / 'data' / 'demo_videos'
RESULT_DIR    = BASE_DIR / 'data' / 'demo_results'
MOBILEVLM_DIR = BASE_DIR / 'MobileVLM'

INTERVAL_SEC  = 2
PROMPT        = """환자가 누워있는지 걷고있는지 행동만 간결하게 설명해. 낙상이 아니면 '낙상'이라는 단어를 절대 꺼내지 마."""
ALERT_KEYWORD = '낙상'

RESULT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(MOBILEVLM_DIR))

print(f'\n타겟 디바이스: {TARGET_DEVICE}')
print(f'병합 모델 존재: {MERGED_DIR.exists()}')


## 🔄 Cell 3: PyTorch → ONNX →  NNCF INT8 모델 변환

> `mobilevlm_lm.onnx` 형태에서 곧바로 NNCF 8-bit 변환을 수행합니다.

In [ ]:
import torch
import openvino as ov
from pathlib import Path

OV_INT8_PATH = OV_DIR / 'mobilevlm_lm_int8.xml'
ONNX_PATH    = OV_DIR / 'mobilevlm_lm.onnx'

if OV_INT8_PATH.exists():
    print(f'✅ OpenVINO INT8 모델 이미 존재: {OV_INT8_PATH}')
    print('   Cell 4로 바로 이동하세요.')
elif not MERGED_DIR.exists():
    print('❌ 병합 모델 없음!')
    print('   먼저 Colab에서 LoRA를 병합하고 다운로드 받으세요.')
else:
    print('PyTorch → ONNX 변환 시작... (약 2~5분 소요)')
    
    from mobilevlm.model.mobilevlm import load_pretrained_model
    from mobilevlm.utils import disable_torch_init

    disable_torch_init()
    tokenizer, model, image_processor, _ = load_pretrained_model(
        str(MERGED_DIR), load_8bit=False, load_4bit=False, device='cpu'
    )
    model = model.float().eval()
    
    # 먼저 ONNX로 변환
    OV_DIR.mkdir(exist_ok=True)
    if not ONNX_PATH.exists():
        dummy_input = torch.zeros(1, 10, dtype=torch.long)
        try:
            torch.onnx.export(
                model.model,
                (dummy_input,),
                str(ONNX_PATH),
                opset_version=14,
                input_names=['input_ids'],
                output_names=['hidden_states'],
                dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'}},
                verbose=False
            )
            print(f'✅ ONNX 형태 저장: {ONNX_PATH}')
        except Exception as e:
            print(f'⚠️ ONNX 변환 오류: {e}')

    # NNCF로 INT8 양자화 적용
    if ONNX_PATH.exists() and not OV_INT8_PATH.exists():
        print('\n🔄 NNCF INT8 양자화 시작 (NPU용 경량화)...')
        import nncf
        
        core = ov.Core()
        ov_model = core.read_model(str(ONNX_PATH))
        
        # NNCF Calibration 데이터를 위한 더미 생성기
        # (진짜 이미지로 하면 더 좋으나 편의를 위해 random id를 사용합니다)
        def transform_fn(data_item):
            dummy_input = torch.randint(0, 32000, (1, 10), dtype=torch.long)
            return {'input_ids': dummy_input.numpy()}
            
        dummy_dataset = [i for i in range(100)]
        calibration_dataset = nncf.Dataset(dummy_dataset, transform_fn)
        
        # 8-bit 양자화
        print('   INT8 스케일 계산 연산 중... (약 1~2분 소요)')
        quantized_model = nncf.quantize(
            ov_model,
            calibration_dataset,
            subset_size=100,
            fast_bias_correction=True
        )
        
        ov.save_model(quantized_model, str(OV_INT8_PATH))
        print(f'✅ INT8 모델 저장 완료: {OV_INT8_PATH}')

## 🧠 Cell 4: NPU 추론 실행

> OpenVINO 변환이 완료된 경우 NPU에서 추론합니다.  
> 변환이 완료되지 않은 경우 PyTorch CPU 모드로 폴백합니다.

In [ ]:
import sys, importlib
if 'mobilevlm.model.mobilellama' in sys.modules: importlib.reload(sys.modules['mobilevlm.model.mobilellama'])
if 'mobilevlm.model.mobilevlm' in sys.modules: importlib.reload(sys.modules['mobilevlm.model.mobilevlm'])

import torch
import openvino as ov
from mobilevlm.model.mobilevlm import load_pretrained_model
from mobilevlm.utils import disable_torch_init, process_images, tokenizer_image_token
from mobilevlm.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from mobilevlm.conversation import conv_templates

OV_MODEL_PATH = OV_DIR / 'mobilevlm_lm_int8.xml'
USE_OV = OV_MODEL_PATH.exists()

if USE_OV:
    print(f'✅ OpenVINO INT8 모델 로드 중...')
    core = ov.Core()
    ov_model = core.read_model(str(OV_MODEL_PATH))

    # 동적 shape 대응: NPU는 제한될 수 있으므로 정적으로 reshape
    if TARGET_DEVICE == 'NPU':
        print(f'   NPU 모드: 모델 Shape 설정 중...')
        try:
             # sequence 길이를 대충 최대 길이로 고정해버림 (NPU 동적 형상 이슈 회피)
            ov_model.reshape({'input_ids': [1, 256]})
            print('   ✅ 정적(Static) Shape으로 컴파일 완료')
        except Exception as e:
            print(f'   ⚠️ 일반 모드 컴파일 시도 ({e})')

    TARGET_DEVICE = 'GPU'
    compiled_model = core.compile_model(ov_model, TARGET_DEVICE)
    infer_req = compiled_model.create_infer_request()
    print(f'✅ {TARGET_DEVICE} 컴파일 준비 완료!')

# 전체 모델도 로드 (비전 인코더 + tokenizer 용)
disable_torch_init()
MODEL_PATH = str(MERGED_DIR) if MERGED_DIR.exists() else 'mtgv/MobileVLM_V2-1.7B'
tokenizer, model_full, image_processor, _ = load_pretrained_model(
    MODEL_PATH, load_8bit=False, load_4bit=False, device='cpu'
)
model_full = model_full.bfloat16().eval()

def format_time(seconds):
    return f'{int(seconds)//60:02d}:{int(seconds)%60:02d}'

def infer_frame(pil_image):
    """이미지 → NPU/OpenVINO 추론 (단순화: 폴백 위주)"""
    image_tensor = process_images([pil_image], image_processor, model_full.config)[0]
    image_tensor = image_tensor.to('cpu', dtype=torch.float32)

    full_prompt = DEFAULT_IMAGE_TOKEN + '\n' + PROMPT
    conv = conv_templates['v1'].copy()
    conv.append_message(conv.roles[0], full_prompt)
    conv.append_message(conv.roles[1], None)

    input_ids = tokenizer_image_token(
        conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0)

    # OpenVINO 또는 PyTorch 폴백으로 추론
    with torch.inference_mode():
        output_ids = model_full.generate(
            input_ids,
            images=image_tensor.unsqueeze(0),
            do_sample=False,                 # 탐색적 생성 끄기(강력한 결정론적 태도 유지)
            max_new_tokens=40,               # 짧게 요약
            use_cache=True,
        )
    return tokenizer.decode(
        output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

print(f'✅ 추론 준비 완료')


## 🎬 Cell 5: 영상 추론 실행

In [ ]:
video_files = sorted([v for v in VIDEO_DIR.iterdir()
                      if v.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv')])

if not video_files:
    print('[ERROR] demo_videos/ 폴더에 영상을 넣어주세요.')
else:
    VIDEO_INDEX = 0
    video_path  = video_files[VIDEO_INDEX]

    cap        = cv2.VideoCapture(str(video_path))
    fps        = cap.get(cv2.CAP_PROP_FPS) or 30
    total_f    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_step = max(1, int(fps * INTERVAL_SEC))

    print(f'\n{"="*60}')
    print(f'📹 {video_path.name} | 디바이스: {TARGET_DEVICE}')
    print(f'{"="*60}')

    report, prev_status, frame_idx, alert_count = [], None, 0, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # 정해진 인터벌(frame_step) 마다만 분석 수행
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        pil_img    = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        time_label = format_time(frame_idx / fps)

        t0      = time.time()
        status  = infer_frame(pil_img)
        elapsed = time.time() - t0

        if status != prev_status or prev_status is None:
            if ALERT_KEYWORD in status: alert_count += 1
            alert_mark = '⚠️ 간호사 호출!' if ALERT_KEYWORD in status else ''
            line  = f'{time_label}  {status[:45]}...  {alert_mark}  [{elapsed:.1f}s]'
            print(line)
            report.append(line)
            prev_status = status
        
        frame_idx += 1

    cap.release()
    
    rp = RESULT_DIR / f'{video_path.stem}_npu_report.txt'
    rp.write_text('\n'.join(report), encoding='utf-8')
    print(f'\n📄 보고서: {rp}')


## 📌 NPU 관련 참고사항

### MobileVLM → NPU 변환의 현실적 제약

MobileVLM V2는 VLM(Vision-Language Model)이라 단순 변환에 한계가 있습니다:

| 컴포넌트 | OpenVINO 변환 가능 여부 |
|---|---|
| Vision Encoder (MobileCLIP) | ✅ 가능 |
| Vision Projector (LDPNetV2) | ⚠️ 커스텀 연산 주의 |
| Language Model (MobileLlama) | ✅ 가능 (별도 변환) |
| 전체 end-to-end | ❌ 파이프라인 구성 필요 |

### 완전한 NPU 배포를 위한 Phase 3 작업
1. 각 컴포넌트를 별도로 ONNX 변환
2. NNCF로 INT8 양자화 적용
3. OpenVINO IR 통합 파이프라인 구성
4. NPU 드라이버 + OpenVINO NPU 플러그인 설치

현재 노트북은 기반 구조를 제공하며, Phase 3에서 완전 구현될 예정입니다.